# Ornith-1.5-35B-A3B — Jalankan llama-server di Kaggle T4x2 + Cloudflare Tunnel

Notebook ini mengasumsikan model GGUF **sudah tersedia sebagai Kaggle Dataset**
(dibuat lewat notebook terpisah `ornith_dataset_builder.ipynb`), dan dataset itu
**sudah di-attach** ke sesi ini lewat menu **Add Data**.

Isi notebook ini:

1. **Bagian 1** — Jalankan `llama-server` (llama.cpp) di Kaggle T4x2 dari model yang sudah ada di dataset,
   dengan otentikasi API key.
2. **Bagian 2** — Expose server ke internet lewat Cloudflare, dengan **named tunnel** (URL tetap)
   sebagai pilihan utama dan **fallback otomatis ke quick tunnel gratis** kalau named tunnel tidak tersedia.


## Bagian 1 — Jalankan llama-server di Kaggle T4x2

Pastikan accelerator notebook diset ke **GPU T4 x2** (Settings -> Accelerator).


In [ ]:
# B0. Bersihkan sisa disk dari percobaan sebelumnya (WAJIB dijalankan kalau pernah kena error "No space left on device")
import shutil, os

paths_to_clean = [
    "/kaggle/working/llama.cpp",       # dari versi cell B1 lama yang salah copy seluruh /kaggle/input
    "/kaggle/working/llama-cpp-bin",   # folder tujuan versi B1 terbaru, dibersihkan supaya mulai dari nol
    "/kaggle/working/ornith-gguf",     # sisa percobaan dataset builder lama (kalau ada)
]

for p in paths_to_clean:
    if os.path.exists(p):
        size_before = sum(f.stat().st_size for f in __import__('pathlib').Path(p).rglob('*') if f.is_file())
        shutil.rmtree(p)
        print(f"Dihapus: {p} (~{size_before / (1024**3):.2f} GB dibebaskan)")
    else:
        print(f"Tidak ada: {p} (skip)")

# Cek sisa disk sekarang
print()
!df -h /kaggle/working


In [ ]:
# B1. Gunakan llama-server hasil kompilasi sendiri (skip build dari source)
# Dataset kamu: baimnot/llama-cpp-cuda-sm75-build
# Pastikan dataset ini sudah di-attach lewat "Add Data" di notebook ini.
#
# Struktur dataset FLAT -- semua file (BUILD_INFO.txt, libggml*.so, libllama*.so,
# llama-cli, llama-server) ada dalam SATU folder yang sama. Jadi cukup salin
# seluruh isi folder itu apa adanya (bukan struktur build/bin/lib terpisah).
# Karena isinya cuma binary + shared library (bukan source tree), ini kecil & cepat.

import glob, os, shutil, stat

# Cari file "llama-server" di dalam dataset yang di-attach.
candidates = glob.glob("/kaggle/input/**/llama-server", recursive=True)
print("llama-server ditemukan di:")
for c in candidates:
    print(" -", c)

if not candidates:
    raise FileNotFoundError(
        "Binary 'llama-server' tidak ditemukan di /kaggle/input. "
        "Pastikan dataset 'llama-cpp-cuda-sm75-build' sudah di-attach via Add Data."
    )

SRC_DIR = os.path.dirname(candidates[0])  # folder flat berisi semua .so + binary

DEST_DIR = "/kaggle/working/llama-cpp-bin"
if os.path.exists(DEST_DIR):
    shutil.rmtree(DEST_DIR)
os.makedirs(DEST_DIR, exist_ok=True)

# Salin SEMUA file dalam folder flat tersebut (binary + seluruh .so pendukung)
copied = []
for fname in os.listdir(SRC_DIR):
    src_path = os.path.join(SRC_DIR, fname)
    if os.path.isfile(src_path):
        dst_path = os.path.join(DEST_DIR, fname)
        shutil.copy2(src_path, dst_path)
        copied.append(fname)

print(f"\n{len(copied)} file disalin ke {DEST_DIR}:")
for f in sorted(copied):
    print(" -", f)

LLAMA_SERVER_BIN = os.path.join(DEST_DIR, "llama-server")

# Kaggle input read-only -> pastikan file hasil copy executable
st = os.stat(LLAMA_SERVER_BIN)
os.chmod(LLAMA_SERVER_BIN, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# Shared library (libggml*.so, libllama*.so, dll) ada SEJAJAR dengan binary di folder yang sama,
# jadi cukup tambahkan folder ini ke LD_LIBRARY_PATH supaya linker runtime bisa menemukannya.
os.environ["LD_LIBRARY_PATH"] = DEST_DIR + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("\nLLAMA_SERVER_BIN siap di:", LLAMA_SERVER_BIN)
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])


In [ ]:
# B2. Cek path model dari dataset yang sudah di-attach
import glob

candidates = glob.glob("/kaggle/input/**/*.gguf", recursive=True)
print("File GGUF ditemukan:")
for c in candidates:
    print(" -", c)

# Set manual sesuai hasil di atas
MODEL_PATH = candidates[0] if candidates else "/kaggle/input/ornith-1-5-35b-a3b-q4km-gguf/Ornith-1.5-35B-Q4_K_M.gguf"
print("\nMenggunakan MODEL_PATH:", MODEL_PATH)


In [ ]:
# B2b. Siapkan API key untuk otentikasi llama-server
# Klien HARUS mengirim header: Authorization: Bearer <API_KEY>
#
# ⚠️ PENTING soal key yang "berubah-ubah":
#   Kalau kamu TIDAK set lewat Kaggle Secret, API_KEY di-generate ULANG SETIAP notebook di-run.
#   Ini sering bikin bingung: kamu paste key dari run sebelumnya ke Web UI, tapi server yang
#   sedang jalan sekarang sudah pakai key BARU -> hasilnya "Invalid API key".
#
# SOLUSI PERMANEN (disarankan):
#   1. Add-ons -> Secrets -> Add Secret
#   2. Label: llama_server_api_key
#   3. Value: bikin sendiri string bebas, misal: sk-ornith-rahasia123 (atau apapun, tidak harus hex)
#   4. Aktifkan toggle-nya untuk notebook ini
#   Dengan begini, key akan SELALU SAMA setiap kali notebook dijalankan ulang.

import secrets as _secrets

API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    _uc = UserSecretsClient()
    API_KEY = _uc.get_secret("llama_server_api_key")
    print("API key dimuat dari Kaggle Secret 'llama_server_api_key' (TETAP, tidak berubah tiap run).")
except Exception:
    API_KEY = _secrets.token_hex(24)  # 48 karakter hex, acak per sesi
    print("Secret 'llama_server_api_key' tidak ditemukan -> generate API key ACAK untuk sesi ini SAJA.")
    print("   Key ini akan BERBEDA lagi kalau notebook di-restart/run ulang -- lihat instruksi di atas")
    print("   untuk membuatnya permanen lewat Kaggle Secrets.")

print("\nAPI_KEY saat ini:", API_KEY)


In [ ]:
# B3. Jalankan llama-server di background (dengan otentikasi API key + context lebih besar)
import subprocess, time

# LLAMA_SERVER_BIN sudah didefinisikan di cell B1
# API_KEY sudah disiapkan di cell B2b

# ---- Konfigurasi context window ----
# Model ini native mendukung hingga 262144 token, tapi VRAM 2x T4 (32GB total) jadi
# batasan nyata setelah weight model (~21.7GB untuk Q4_K_M) dimuat.
#
# Untuk memuat context lebih besar tanpa OOM, KV cache di-KUANTISASI ke Q8_0
# (--cache-type-k / --cache-type-v), yang butuh --flash-attn diaktifkan. Ini bisa
# menghemat KV cache sampai ~50% dibanding f16 default, sehingga context bisa dinaikkan
# jauh lebih tinggi dari 8192 sebelumnya.

CONTEXT_LENGTH = 131072  # naikkan/turunkan sesuai kebutuhan & hasil test di bawah
# Alternatif untuk dicoba kalau VRAM masih longgar: 65536, 32768, 131072
# Turunkan ke 16384 atau 8192 kalau masih OOM di 32768.

server_proc = subprocess.Popen([
    LLAMA_SERVER_BIN,
    "-m", MODEL_PATH,
    "--host", "0.0.0.0",
    "--port", "8080",
    "-ngl", "999",              # offload semua layer yang muat ke GPU
    "--tensor-split", "1,1",    # split rata ke 2x T4
    "-c", str(CONTEXT_LENGTH),  # context window
    "--flash-attn", "on",       # wajib untuk KV cache quantization di bawah
    "--cache-type-k", "q8_0",   # kuantisasi KV cache (K) -> hemat VRAM signifikan
    "--cache-type-v", "q8_0",   # kuantisasi KV cache (V) -> hemat VRAM signifikan
    "--api-key", API_KEY,       # klien wajib kirim: Authorization: Bearer <API_KEY>
])

time.sleep(25)  # tunggu server siap load model (lebih lama karena context lebih besar)
print(f"llama-server berjalan di http://localhost:8080 (context={CONTEXT_LENGTH}, KV cache Q8_0, otentikasi aktif)")
print("\nKalau proses di atas mati/error (cek dengan cell B4), kemungkinan OOM --")
print("turunkan CONTEXT_LENGTH di cell ini, lalu jalankan ulang.")


In [ ]:
# B4. (Opsional) Cek server sudah hidup
import requests

# Cek dulu apakah proses server masih hidup (kalau OOM karena context terlalu besar,
# proses akan mati dan poll() mengembalikan return code, bukan None)
if server_proc.poll() is not None:
    print(f"⚠️  Proses llama-server SUDAH MATI (exit code: {server_proc.returncode}).")
    print("Kemungkinan OOM karena CONTEXT_LENGTH terlalu besar untuk VRAM yang tersedia.")
    print("-> Turunkan CONTEXT_LENGTH di cell B3, lalu jalankan ulang cell B3 dan B4.")
    out, _ = server_proc.communicate()
    if out:
        print("\nLog terakhir:\n", out[-2000:])  # tampilkan 2000 karakter terakhir log
else:
    headers = {"Authorization": f"Bearer {API_KEY}"}
    try:
        r = requests.get("http://localhost:8080/health", headers=headers, timeout=5)
        print(r.status_code, r.text)
    except Exception as e:
        print("Server belum siap / error:", e)


## Bagian 2 — Cloudflare Tunnel (Named dengan fallback Quick Tunnel gratis)

Berbeda dari quick tunnel (`trycloudflare.com`) yang URL-nya acak dan sementara,
**named tunnel** memberi kamu subdomain tetap di bawah domain kamu sendiri yang sudah terdaftar di Cloudflare.

**Prasyarat sebelum menjalankan Bagian C:**
1. Domain sudah ditambahkan & aktif (proxied) di akun Cloudflare kamu.
2. Sudah membuat **Cloudflare API Token** atau melakukan `cloudflared login` sekali untuk mendapatkan **origin certificate** (`cert.pem`).
   - Karena Kaggle adalah environment non-interaktif/headless, cara termudah adalah:
     a. Jalankan `cloudflared tunnel login` di komputer lokal kamu (akan membuka browser untuk otorisasi).
     b. Ambil file `cert.pem` yang dihasilkan (biasanya di `~/.cloudflared/cert.pem`).
     c. Upload isi `cert.pem` tersebut sebagai **Kaggle Secret** bernama `cf_cert_pem`.
3. Simpan juga token tunnel (lihat cell C3) sebagai Kaggle Secret bernama `cf_tunnel_token` **setelah** tunnel dibuat sekali (lihat catatan di cell C3).

Kalau kamu belum pernah membuat tunnel sama sekali, jalankan cell C2 & C3 di bawah **secara lokal terlebih dahulu**
(bukan di Kaggle) untuk generate `TUNNEL_ID` dan token, karena proses `login` butuh browser interaktif.
Setelah punya `cf_tunnel_token`, seluruh proses berikutnya bisa 100% dijalankan di Kaggle tanpa browser.


In [ ]:
# C1. Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


### C2 & C3 — dijalankan SEKALI di komputer lokal (bukan di Kaggle)

Ini contoh perintah yang kamu jalankan di terminal lokal kamu (Mac/Linux/WSL) untuk membuat named tunnel.
Salin hasilnya (terutama **token**) untuk dipakai kembali di Kaggle lewat Secrets.

```bash
# Login (membuka browser, pilih domain Cloudflare kamu)
cloudflared tunnel login

# Buat tunnel baru dengan nama bebas, misal "ornith-llm"
cloudflared tunnel create ornith-llm
# -> ini akan menghasilkan TUNNEL_ID dan file kredensial JSON di ~/.cloudflared/<TUNNEL_ID>.json

# Arahkan subdomain ke tunnel ini, misal llm.domainkamu.com
cloudflared tunnel route dns ornith-llm llm.domainkamu.com

# Dapatkan token untuk menjalankan tunnel tanpa perlu file cert/kredensial lagi
cloudflared tunnel token ornith-llm
# -> salin output token ini, simpan sebagai Kaggle Secret dengan nama: cf_tunnel_token
```

Setelah punya token, kamu **tidak perlu lagi** cert.pem atau file kredensial JSON di Kaggle —
cukup token tersebut untuk menjalankan tunnel dari sel C4 di bawah.


In [ ]:
# C4. Jalankan tunnel: coba NAMED TUNNEL dulu, fallback ke QUICK TUNNEL gratis jika gagal
import os, subprocess, time, re

CF_TUNNEL_TOKEN = None
tunnel_mode = None       # "named" atau "quick"
PUBLIC_URL = None        # akan diisi otomatis untuk mode quick; untuk mode named, isi manual di C5

# 1) Coba ambil token named tunnel dari Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    CF_TUNNEL_TOKEN = secrets.get_secret("cf_tunnel_token")
    print("Token 'cf_tunnel_token' ditemukan, mencoba NAMED TUNNEL...")
except Exception as e:
    print("Secret 'cf_tunnel_token' tidak ditemukan/tidak ter-attach.")
    print("  ->", e)

tunnel_proc = None

if CF_TUNNEL_TOKEN:
    # ---- Coba jalankan NAMED TUNNEL ----
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "run", "--token", CF_TUNNEL_TOKEN, "--url", "http://localhost:8080"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    time.sleep(8)
    if tunnel_proc.poll() is None:
        # proses masih hidup -> anggap named tunnel berhasil start
        tunnel_mode = "named"
        print("Named tunnel berjalan. Cek log:")
        for _ in range(10):
            line = tunnel_proc.stdout.readline()
            if not line:
                break
            print(line.strip())
        print("\n(Named tunnel aktif. Set PUBLIC_URL manual di cell C5 sesuai subdomain yang kamu route.)")
    else:
        # proses named tunnel mati / error -> fallback
        out, _ = tunnel_proc.communicate()
        print("Named tunnel GAGAL start, keluar dengan output:")
        print(out)
        print("\nFallback ke QUICK TUNNEL gratis...")
        CF_TUNNEL_TOKEN = None  # trigger fallback di bawah

if not CF_TUNNEL_TOKEN:
    # ---- Fallback: QUICK TUNNEL gratis (trycloudflare.com) ----
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8080"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    url = None
    for line in tunnel_proc.stdout:
        print(line.strip())
        match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break

    if url:
        tunnel_mode = "quick"
        PUBLIC_URL = url
        print("\n🔗 Quick tunnel aktif (gratis, URL sementara):", PUBLIC_URL)
    else:
        print("\nGagal mendapatkan URL quick tunnel. Cek log cloudflared di atas.")

print("\nMode tunnel yang aktif:", tunnel_mode)


In [ ]:
# C5. Cek endpoint publik kamu (dengan otentikasi API key)
import requests

if tunnel_mode == "named":
    # Named tunnel: isi manual subdomain yang sudah kamu route lewat `cloudflared tunnel route dns`
    PUBLIC_URL = "https://llm.domainkamu.com"  # <-- ganti sesuai subdomain kamu
elif tunnel_mode == "quick":
    # PUBLIC_URL sudah otomatis terisi dari cell C4 (trycloudflare.com)
    pass
else:
    raise RuntimeError("Tidak ada tunnel yang berhasil dijalankan. Cek log di cell C4.")

print(f"Mode tunnel: {tunnel_mode}")
print("Server llama.cpp kamu (OpenAI-compatible API) dapat diakses di:")
print(f"  Chat completions : {PUBLIC_URL}/v1/chat/completions")
print(f"  Health check      : {PUBLIC_URL}/health")
print(f"\nAPI key (wajib disertakan klien via header Authorization): {API_KEY}")
print("Contoh header yang harus dikirim klien:")
print(f'  Authorization: Bearer {API_KEY}')

headers = {"Authorization": f"Bearer {API_KEY}"}

try:
    r = requests.get(f"{PUBLIC_URL}/health", headers=headers, timeout=10)
    print("\nStatus (dengan API key):", r.status_code, r.text)
except Exception as e:
    print("\nBelum bisa diakses dari sini (DNS propagation mungkin butuh waktu):", e)

# Contoh request tanpa API key -> harus ditolak (401)
try:
    r_noauth = requests.get(f"{PUBLIC_URL}/health", timeout=10)
    print("\nStatus TANPA API key (harus 401):", r_noauth.status_code)
except Exception as e:
    print("\nGagal cek tanpa API key:", e)


# ---- Ringkasan siap-copy (jalankan cell ini lagi kapan saja untuk lihat nilai TERKINI) ----
print("\n" + "="*60)
print("RINGKASAN KONEKSI TERKINI (pakai nilai ini, jangan pakai dari run sebelumnya)")
print("="*60)
print(f"URL   : {PUBLIC_URL}")
print(f"KEY   : {API_KEY}")
print("="*60)
print("Di Web UI llama.cpp -> Settings -> API Key, paste KEY di atas TANPA kata 'Bearer' dan tanpa spasi/enter tambahan.")
print("Kalau quick tunnel (gratis): URL ini akan MATI setelah kernel/cell C4 di-restart -- jalankan ulang C4 & C5 untuk dapat URL baru.")


In [ ]:
# C6. Generate contoh curl siap-pakai (otomatis pakai URL & KEY yang sedang aktif)

curl_health = f'''curl -s {PUBLIC_URL}/health \\
  -H "Authorization: Bearer {API_KEY}"'''

curl_chat = f'''curl -s {PUBLIC_URL}/v1/chat/completions \\
  -H "Authorization: Bearer {API_KEY}" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "model": "ornith-1.5-35b-a3b",
    "messages": [
      {{"role": "user", "content": "Tulis fungsi Python untuk cek bilangan prima."}}
    ],
    "max_tokens": 512,
    "temperature": 0.6
  }}' '''

curl_stream = f'''curl -N -s {PUBLIC_URL}/v1/chat/completions \\
  -H "Authorization: Bearer {API_KEY}" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "model": "ornith-1.5-35b-a3b",
    "messages": [{{"role": "user", "content": "Halo, kamu siapa?"}}],
    "stream": true
  }}' '''

curl_noauth = f'''curl -i {PUBLIC_URL}/health'''

print("="*70)
print("1) HEALTH CHECK (dengan API key)")
print("="*70)
print(curl_health)

print("\n" + "="*70)
print("2) CHAT COMPLETION (non-streaming)")
print("="*70)
print(curl_chat)

print("\n" + "="*70)
print("3) CHAT COMPLETION (streaming)")
print("="*70)
print(curl_stream)

print("\n" + "="*70)
print("4) TANPA API KEY (harus dapat 401)")
print("="*70)
print(curl_noauth)

print("\n\nCopy salah satu command di atas, jalankan dari terminal lokal kamu (bukan di sini).")


## Catatan Penting

- **Named tunnel** memerlukan proses login interaktif satu kali **di luar Kaggle** (karena Kaggle headless).
  Setelah token didapat, semua proses berikutnya sepenuhnya otomatis di Kaggle.
- Simpan token dan kredensial (`cf_tunnel_token`) sebagai **Kaggle Secrets**, jangan hardcode di notebook —
  supaya aman saat notebook di-share atau dipublikasikan.
- Sesi Kaggle punya batas waktu (idle timeout, max runtime). Named tunnel tetap akan mati saat sesi Kaggle berhenti;
  bedanya dengan quick tunnel hanya di URL yang **konsisten** setiap kali kamu jalankan ulang, bukan soal keep-alive 24/7.
- Untuk keperluan produksi 24/7 sebaiknya host llama-server di server yang selalu menyala
  (VPS/GPU cloud), Kaggle T4x2 lebih cocok untuk eksperimen/testing.
- Sesuaikan `-c` (context length) dan `--tensor-split` di Bagian B sesuai kebutuhan dan sisa VRAM
  setelah model dimuat, untuk menghindari OOM di 2x T4 16GB.
